In [2]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [1]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Colab Notebooks/TFM")

NORMALIZADOS = BASE / "Normalizados paraextraer"
CAPITULOS = BASE / "Capítulos"

GRAFOS = BASE / "Grafos"
EDGES = GRAFOS / "Edges"
NODES = GRAFOS / "Nodes"
RESULTADOS = GRAFOS / "Resultados"

for carpeta in [GRAFOS, EDGES, NODES, RESULTADOS]:
    carpeta.mkdir(parents=True, exist_ok=True)

print("Rutas listas")
print("Normalizados:", NORMALIZADOS.exists())
print("Capítulos:", CAPITULOS.exists())

Rutas listas
Normalizados: False
Capítulos: True


In [ ]:
import re
import unicodedata
from collections import Counter
from pathlib import Path
import pandas as pd
import shutil

from spacy.lang.es.stop_words import STOP_WORDS

stopwords_extra = {
    "si","sí","ya","tan","solo","sólo","aun","aún","muy",
    "porque","pues","entonces","aquí","allí","también","pero",
    "dijo","dice","hacer","hace","así","ser","estar"
}

STOPWORDS = set(STOP_WORDS).union(stopwords_extra)

def quitar_tildes(texto):
    return ''.join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

# normalizamos también stopwords sin tilde
STOPWORDS = {quitar_tildes(w.lower()) for w in STOPWORDS}

def tokenizar(texto):
    texto = quitar_tildes(texto.lower())
    texto = re.sub(r"[^a-zñ]+", " ", texto)

    tokens = texto.split()

    tokens = [
        t for t in tokens
        if t not in STOPWORDS
        and len(t) > 2
    ]

    return tokens

def crear_edges(tokens, ventana=5):
    edges = Counter()

    for i, palabra in enumerate(tokens):
        limite = min(i + ventana + 1, len(tokens))

        for j in range(i + 1, limite):
            a, b = sorted([palabra, tokens[j]])
            edges[(a, b)] += 1

    return edges

print("Funciones con spaCy STOP_WORDS listas")

Funciones con spaCy STOP_WORDS listas


In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Colab Notebooks/TFM")
CAPITULOS = BASE / "Capítulos"
RESULTADOS = BASE / "Grafos" / "Resultados"
RESULTADOS.mkdir(parents=True, exist_ok=True)

pares = [
    (1, 11, "capitulos_libro1", "capitulos_libro11"),
    (2, 12, "capitulos_libro2", "capitulos_libro12"),
    (3, 13, "capitulos_libro3", "capitulos_libro13"),
    (4, 14, "capitulos_libro4", "capitulos_libro14"),
    (5, 15, "capitulos_libro5", "capitulos_libro15"),
    (6, 16, "bloques_libro6", "bloques_libro16"),
    (7, 17, "bloques_libro7", "bloques_libro17"),
    (8, 18, "bloques_libro8", "bloques_libro18"),
    (9, 19, "Capitulos_libro9", "Capitulos_libro19"),
    (10, 20, "bloques_libro10", "bloques_libro20"),
]

def jaccard(a, b):
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)

resultados = []

for libro_a, libro_b, carpeta_a, carpeta_b in pares:
    dir_a = CAPITULOS / carpeta_a
    dir_b = CAPITULOS / carpeta_b

    archivos_a = sorted(dir_a.glob("*.txt"))
    archivos_b = sorted(dir_b.glob("*.txt"))

    if len(archivos_a) != len(archivos_b):
        print(f"REVISAR {libro_a}-{libro_b}: {len(archivos_a)} vs {len(archivos_b)}")
        continue

    for idx, (fa, fb) in enumerate(zip(archivos_a, archivos_b), start=1):
        texto_a = fa.read_text(encoding="utf-8", errors="ignore")
        texto_b = fb.read_text(encoding="utf-8", errors="ignore")

        tokens_a = tokenizar(texto_a)
        tokens_b = tokenizar(texto_b)

        vocab_a = set(tokens_a)
        vocab_b = set(tokens_b)

        edges_a = crear_edges(tokens_a, ventana=5)
        edges_b = crear_edges(tokens_b, ventana=5)

        set_edges_a = set(edges_a.keys())
        set_edges_b = set(edges_b.keys())

        resultados.append({
            "pareja": f"{libro_a}-{libro_b}",
            "libro_a": libro_a,
            "libro_b": libro_b,
            "unidad": idx,
            "archivo_a": fa.name,
            "archivo_b": fb.name,
            "tokens_a": len(tokens_a),
            "tokens_b": len(tokens_b),
            "vocab_a": len(vocab_a),
            "vocab_b": len(vocab_b),
            "edges_a": len(set_edges_a),
            "edges_b": len(set_edges_b),
            "jaccard_vocab": jaccard(vocab_a, vocab_b),
            "jaccard_edges": jaccard(set_edges_a, set_edges_b),
            "dif_tokens_abs": abs(len(tokens_a) - len(tokens_b)),
            "dif_edges_abs": abs(len(set_edges_a) - len(set_edges_b)),
        })

df = pd.DataFrame(resultados)

salida = RESULTADOS / "comparacion_unidades_jaccard_edges.csv"
df.to_csv(salida, index=False, encoding="utf-8-sig")

print("Filas:", len(df))
print("Guardado en:", salida)
df.head()

Filas: 0
Guardado en: /content/drive/MyDrive/Colab Notebooks/TFM/Grafos/Resultados/comparacion_unidades_jaccard_edges.csv


""


In [ ]:
import pandas as pd

ruta = RESULTADOS / "comparacion_unidades_jaccard_edges.csv"

df = pd.read_csv(ruta)

print("Filas:", len(df))
df.head()

Filas: 155


,pareja,libro_a,libro_b,unidad,archivo_a,archivo_b,tokens_a,tokens_b,vocab_a,vocab_b,edges_a,edges_b,jaccard_vocab,jaccard_edges,dif_tokens_abs,dif_edges_abs
0,1-11,1,11,1,00_Preámbulo.txt,00_Preámbulo.txt,3573,3568,1675,1681,15601,15607,0.970640,0.952208,5,6
1,1-11,1,11,2,01_I.txt,01_I.txt,1057,1057,655,653,4987,4988,0.972851,0.952437,0,1
2,1-11,1,11,3,02_II.txt,02_II.txt,1116,1120,707,704,5314,5323,0.978962,0.961099,4,9
3,1-11,1,11,4,03_III.txt,03_III.txt,1040,1042,638,640,4805,4813,0.984472,0.970498,2,8
4,1-11,1,11,5,04_IV.txt,04_IV.txt,1150,1150,711,710,5438,5440,0.970874,0.947368,0,2


In [ ]:
cols = [
    "pareja",
    "unidad",
    "archivo_a",
    "archivo_b",
    "jaccard_vocab",
    "jaccard_edges",
    "dif_edges_abs"
]

df.sort_values(
    by="jaccard_edges",
    ascending=True
)[cols].head(20)

,pareja,unidad,archivo_a,archivo_b,jaccard_vocab,jaccard_edges,dif_edges_abs
102,6-16,14,bloque_14.txt,bloque_14.txt,0.073727,0.001735,1667
95,6-16,7,bloque_07.txt,bloque_07.txt,0.095662,0.003474,858
99,6-16,11,bloque_11.txt,bloque_11.txt,0.060714,0.004467,1248
97,6-16,9,bloque_09.txt,bloque_09.txt,0.067358,0.004635,1436
96,6-16,8,bloque_08.txt,bloque_08.txt,0.075243,0.004847,1206
104,6-16,16,bloque_16.txt,bloque_16.txt,0.081911,0.004978,888
107,6-16,19,bloque_19.txt,bloque_19.txt,0.088254,0.005444,4685
98,6-16,10,bloque_10.txt,bloque_10.txt,0.083218,0.005590,1739
105,6-16,17,bloque_17.txt,bloque_17.txt,0.069652,0.005613,268
103,6-16,15,bloque_15.txt,bloque_15.txt,0.070946,0.005689,925


In [ ]:
for n in [1,7,14,20,31]:

    a = (CAPITULOS/"bloques_libro6"/f"bloque_{n:02d}.txt").read_text(
        encoding="utf-8",
        errors="ignore"
    )

    b = (CAPITULOS/"bloques_libro16"/f"bloque_{n:02d}.txt").read_text(
        encoding="utf-8",
        errors="ignore"
    )

    print("\nBLOQUE", n)
    print("6 :", " ".join(a.split()[:25]))
    print("16:", " ".join(b.split()[:25]))


BLOQUE 1
6 : Cap. 1. Yo BIEN SÉ POR QUÉ SALTAS, MI PEQUEÑO ELIACIM Cap. 1. Yo BIEN SÉ POR QUÉ SALTAS, MI PEQUEÑO ELIACIM Venías dando saltos
16: CAMILO JOSE CELA MRS. CALDWELL HABLA CON SU HIJO ADVERTENCIA Conocí a Mrs. Caldwell en Pastrana, durante el viaje que hice por la Alcarria, hace

BLOQUE 7
6 : Cap. 39. El estío. Cap. 39. El estío. El estío es la estación de los moribundos, la estación en la que los moribundos se suben,
16: es, exactamente, lunes. A veces me siento morir, los lunes sobre todo, invadida ior un inconcreto y fortísimo deseo de vivir tres o cuatro días

BLOQUE 14
6 : Cap. 81. El tañedor de acordeón. Cap. 81. El tañedor de acordeón. Yo conocí un tañedor de acordeón leproso que llevaba los gusanitos de sus
16: familia. Para ti, también. Se conoce que no estábamos destinados por la Providencia a hacer vida en familia, a quedarnos en la sobremesa hablando y

BLOQUE 20
6 : Cap. 131. Las más tiernas praderas. Cap. 131. Las más tiernas praderas. Sobre las praderas más ti

In [ ]:
for libro in [6,16]:

    ruta = NORMALIZADOS / f"Libro {libro}_norm.txt"
    texto = ruta.read_text(encoding="utf-8", errors="ignore")

    print("\nLIBRO", libro)
    print("ADVERTENCIA:", texto.lower().count("advertencia"))
    print("Primeros 800 caracteres:\n")
    print(texto[:800])


LIBRO 6
ADVERTENCIA: 2
Primeros 800 caracteres:

﻿CAMILO JOSÉ CELA
MBS. CALDWELL HABLA CON SU HIJO
ADVERTENCIA
CONOCÍ a Mrs. Caldwell en Pastrana, durante el viaje que hice por la Alcarria, hace ya algún tiempo. Mrs. Caldwell estaba despegando con todo cuidado los baldosines de la alcoba donde murió la princesa de Éboli; después los envolvía en papel de seda, uno por uno, y los guardaba en la maleta, una maleta de vientre vario y meticuloso.
En la fonda, Mrs. Caldwell me leyó un día, después de cenar, las páginas que estaba escribiendo en recuerdo de su adorado hijo Eliacim,. tierno como la hoja del culantrillo, muerto heroicamente en las procelosas aguas del Mar Egeo. La obrita de Mrs. Caldwell se titulaba, en principio, “Hablo con mi bienamado hijo Eliacim”. Tenía varios títulos más en cartera, pero, sin duda, el más hermoso era el que

LIBRO 16
ADVERTENCIA: 2
Primeros 800 caracteres:

﻿CAMILO JOSE CELA
MRS. CALDWELL HABLA CON SU HIJO
ADVERTENCIA
Conocí a Mrs. Caldwell en Pastrana, 

In [ ]:
import re
import shutil
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Colab Notebooks/TFM")
NORMALIZADOS = BASE / "Normalizados paraextraer"
CAPITULOS = BASE / "Capítulos"

rangos = [
    (1,16),
    (17,41),
    (42,67),
    (68,91),
    (92,114),
    (115,129),
    (130,146),
    (147,164),
    (165,180),
    (181,197),
    (198,212)
]

for libro in [6,16]:

    texto = (NORMALIZADOS / f"Libro {libro}_norm.txt").read_text(
        encoding="utf-8",
        errors="ignore"
    )

    destino = CAPITULOS / f"bloques_libro{libro}_v2"

    shutil.rmtree(destino, ignore_errors=True)
    destino.mkdir(parents=True, exist_ok=True)

    # localizar capítulos
    patron = re.compile(
        r"(Cap\.\s*(\d+)\..*?)(?=Cap\.\s*\d+\.|$)",
        re.S
    )

    caps = patron.findall(texto)

    caps_dict = {
        int(num): contenido
        for contenido, num in caps
    }

    # advertencias
    primera_adv = texto.split("Cap. 1.")[0]

    pos_ultima_adv = texto.lower().rfind("advertencia")
    ultima_adv = texto[pos_ultima_adv:] if pos_ultima_adv > 0 else ""

    for i, (ini, fin) in enumerate(rangos, start=1):

        bloque = ""

        if i == 1:
            bloque += primera_adv + "\n"

        for n in range(ini, fin+1):
            if n in caps_dict:
                bloque += caps_dict[n] + "\n"

        if i == len(rangos):
            bloque += "\n" + ultima_adv

        salida = destino / f"bloque_{i:02d}.txt"
        salida.write_text(bloque, encoding="utf-8")

    print(f"Libro {libro}: {len(rangos)} bloques creados")
    print(destino)

Libro 6: 11 bloques creados
/content/drive/MyDrive/Colab Notebooks/TFM/Capítulos/bloques_libro6_v2
Libro 16: 11 bloques creados
/content/drive/MyDrive/Colab Notebooks/TFM/Capítulos/bloques_libro16_v2


In [ ]:
for libro in [6,16]:

    carpeta = CAPITULOS / f"bloques_libro{libro}_v2"

    print("\nLIBRO", libro)

    for a in sorted(carpeta.glob("*.txt")):
        texto = a.read_text(encoding="utf-8", errors="ignore")
        inicio = " ".join(texto.split()[:25])
        print(a.name, "->", inicio[:180])


LIBRO 6
bloque_01.txt -> ﻿CAMILO JOSÉ CELA MBS. CALDWELL HABLA CON SU HIJO ADVERTENCIA CONOCÍ a Mrs. Caldwell en Pastrana, durante el viaje que hice por la Alcarria, hace
bloque_02.txt -> Cap. 17. El vecino bien educado. Tú, no. Pero tu primo Ricardo, ¡ah, tu primo Ricardo! Tu primo Ricardo es un zángano que no hace
bloque_03.txt -> Cap. 42. Lord Macaulay. Siempre te amé mucho, hijo mío, siempre te distinguí con mi más egoísta y sincero cariño, siempre has sido para mí
bloque_04.txt -> Cap. 68. Abalorios finos Tengo un traje bordado con cuentas azules y otro con cuentas verdes. Según los pensamientos con que me levanto de la
bloque_05.txt -> Cap. 92. Las gafas ahumadas. Con tus gafas ahumadas para el sol, Eliacim, con aquellas gafas con las que estabas tan gracioso como un francés,
bloque_06.txt -> Cap. 115. El abogado sin pleitos. Yo le llevé mi pleito, Eliacim, a aquel abogado sin pleitos porque pensé que tendría más tiempo libre. El
bloque_07.txt -> Cap. 130.TU VIAJE DE PRÁCTICAS. 

In [ ]:
import re

texto16 = (NORMALIZADOS / "Libro 16_norm.txt").read_text(
    encoding="utf-8",
    errors="ignore"
)

# buscar líneas posibles de inicio de capítulo
patrones = [
    r"(?m)^Cap\.\s*\d+.*$",
    r"(?m)^\d+\.\s+.*$",
    r"(?m)^\d+\s+.*$",
    r"(?m)^Cap[ií]tulo\s+\d+.*$",
]

for patron in patrones:
    matches = re.findall(patron, texto16)
    print("\nPATRÓN:", patron)
    print("Coincidencias:", len(matches))
    print(matches[:10])


PATRÓN: (?m)^Cap\.\s*\d+.*$
Coincidencias: 0
[]

PATRÓN: (?m)^\d+\.\s+.*$
Coincidencias: 213
['1.\tYO BIEN SÉ POR QUÉ SALTAS, MI PEQUEÑO ELIACIM', '2.\t«Music Hall»', '3.\tAyúdame a devanar esta madeja DE LANA DE COLOR CICLAMEN', '4.\tUn tango de los viejos tiempos', '5.\tLa tradición', '6.\tUna partida de poker en la que no media interés', '7.\tLO MALO VINO DESPUÉS', '8.\tEl coñac y el ron', '9.\tEl caballito del diablo', '10.\tLos puros de La Habana']

PATRÓN: (?m)^\d+\s+.*$
Coincidencias: 1
['99\tLos PERROS DE LUJO']

PATRÓN: (?m)^Cap[ií]tulo\s+\d+.*$
Coincidencias: 0
[]


In [ ]:
import re
import shutil
from pathlib import Path

rangos = [
    (1,16),
    (17,41),
    (42,67),
    (68,91),
    (92,114),
    (115,129),
    (130,146),
    (147,164),
    (165,180),
    (181,197),
    (198,212)
]

libro = 16

texto = (NORMALIZADOS / f"Libro {libro}_norm.txt").read_text(
    encoding="utf-8",
    errors="ignore"
)

destino = CAPITULOS / f"bloques_libro{libro}_v2"

shutil.rmtree(destino, ignore_errors=True)
destino.mkdir(parents=True, exist_ok=True)

patron = re.compile(
    r"(?m)^(\d+)\.\s+.*$"
)

matches = list(patron.finditer(texto))

caps_dict = {}

for i, match in enumerate(matches):
    num = int(match.group(1))
    start = match.start()
    end = matches[i+1].start() if i+1 < len(matches) else len(texto)

    caps_dict[num] = texto[start:end].strip()

print("Capítulos detectados:", len(caps_dict))
print("Primeros:", sorted(caps_dict.keys())[:10])
print("Últimos:", sorted(caps_dict.keys())[-10:])

# advertencias
primera_adv = texto[:matches[0].start()] if matches else ""

pos_ultima_adv = texto.lower().rfind("advertencia")
ultima_adv = texto[pos_ultima_adv:] if pos_ultima_adv > matches[-1].start() else ""

for i, (ini, fin) in enumerate(rangos, start=1):

    bloque = ""

    if i == 1:
        bloque += primera_adv + "\n"

    for n in range(ini, fin + 1):
        if n in caps_dict:
            bloque += caps_dict[n] + "\n"

    if i == len(rangos):
        bloque += "\n" + ultima_adv

    (destino / f"bloque_{i:02d}.txt").write_text(
        bloque.strip(),
        encoding="utf-8"
    )

print("Bloques creados:", len(list(destino.glob('*.txt'))))

Capítulos detectados: 211
Primeros: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Últimos: [203, 204, 205, 206, 207, 208, 209, 210, 211, 212]
Bloques creados: 11


In [ ]:
for n in [1,2,5,11]:
    a = (CAPITULOS/"bloques_libro6_v2"/f"bloque_{n:02d}.txt").read_text(encoding="utf-8", errors="ignore")
    b = (CAPITULOS/"bloques_libro16_v2"/f"bloque_{n:02d}.txt").read_text(encoding="utf-8", errors="ignore")

    print("\nBLOQUE", n)
    print("6 :", " ".join(a.split()[:20]))
    print("16:", " ".join(b.split()[:20]))


BLOQUE 1
6 : ﻿CAMILO JOSÉ CELA MBS. CALDWELL HABLA CON SU HIJO ADVERTENCIA CONOCÍ a Mrs. Caldwell en Pastrana, durante el viaje que
16: ﻿CAMILO JOSE CELA MRS. CALDWELL HABLA CON SU HIJO ADVERTENCIA Conocí a Mrs. Caldwell en Pastrana, durante el viaje que

BLOQUE 2
6 : Cap. 17. El vecino bien educado. Tú, no. Pero tu primo Ricardo, ¡ah, tu primo Ricardo! Tu primo Ricardo es
16: 17. El vecino bien educado Tú, no. Pero tu primo Ricardo, ¡ah, tu primo Ricardo! Tu primo Ricardo es un

BLOQUE 5
6 : Cap. 92. Las gafas ahumadas. Con tus gafas ahumadas para el sol, Eliacim, con aquellas gafas con las que estabas
16: 92. Las gafas ahumadas Con tus gafas ahumadas para el sol, Eliacim, con aquellas gafas con las que estabas tan

BLOQUE 11
6 : Cap. 198. Los ENFERMOS DEL HOSPITAL. 1, ¡Qué silenciosamente, qué temerosamente alegres, Eliacim, los enfermos, los hombres y las mujeres
16: 198. Los ENFERMOS DEL HOSPITAL ¡ Qué silenciosamente, que temerosamente alegres, Eliacim, los enfermos, los hombres 

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Colab Notebooks/TFM")
CAPITULOS = BASE / "Capítulos"
RESULTADOS = BASE / "Grafos" / "Resultados"

pares = [
    (1,11,"capitulos_libro1","capitulos_libro11"),
    (2,12,"capitulos_libro2","capitulos_libro12"),
    (3,13,"capitulos_libro3","capitulos_libro13"),
    (4,14,"capitulos_libro4","capitulos_libro14"),
    (5,15,"capitulos_libro5","capitulos_libro15"),

    # VERSION CORREGIDA
    (6,16,"bloques_libro6_v2","bloques_libro16_v2"),

    (7,17,"bloques_libro7","bloques_libro17"),
    (8,18,"bloques_libro8","bloques_libro18"),
    (9,19,"Capitulos_libro9","Capitulos_libro19"),
    (10,20,"bloques_libro10","bloques_libro20"),
]

def jaccard(a,b):
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)

resultados = []

for libro_a, libro_b, carpeta_a, carpeta_b in pares:

    dir_a = CAPITULOS / carpeta_a
    dir_b = CAPITULOS / carpeta_b

    archivos_a = sorted(dir_a.glob("*.txt"))
    archivos_b = sorted(dir_b.glob("*.txt"))

    if len(archivos_a) != len(archivos_b):
        print(f"REVISAR {libro_a}-{libro_b}: {len(archivos_a)} vs {len(archivos_b)}")
        continue

    for idx, (fa, fb) in enumerate(zip(archivos_a, archivos_b), start=1):

        texto_a = fa.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        texto_b = fb.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        tokens_a = tokenizar(texto_a)
        tokens_b = tokenizar(texto_b)

        vocab_a = set(tokens_a)
        vocab_b = set(tokens_b)

        edges_a = crear_edges(tokens_a, ventana=5)
        edges_b = crear_edges(tokens_b, ventana=5)

        set_edges_a = set(edges_a.keys())
        set_edges_b = set(edges_b.keys())

        resultados.append({
            "pareja": f"{libro_a}-{libro_b}",
            "unidad": idx,
            "archivo_a": fa.name,
            "archivo_b": fb.name,
            "tokens_a": len(tokens_a),
            "tokens_b": len(tokens_b),
            "vocab_a": len(vocab_a),
            "vocab_b": len(vocab_b),
            "edges_a": len(set_edges_a),
            "edges_b": len(set_edges_b),
            "jaccard_vocab": jaccard(vocab_a, vocab_b),
            "jaccard_edges": jaccard(set_edges_a, set_edges_b),
            "dif_tokens_abs": abs(len(tokens_a)-len(tokens_b)),
            "dif_edges_abs": abs(len(set_edges_a)-len(set_edges_b)),
        })

df = pd.DataFrame(resultados)

salida = RESULTADOS / "comparacion_unidades_jaccard_edges.csv"

df.to_csv(
    salida,
    index=False,
    encoding="utf-8-sig"
)

print("Filas:", len(df))
print("Guardado en:", salida)

df.head()

Filas: 135
Guardado en: /content/drive/MyDrive/Colab Notebooks/TFM/Grafos/Resultados/comparacion_unidades_jaccard_edges.csv


,pareja,unidad,archivo_a,archivo_b,tokens_a,tokens_b,vocab_a,vocab_b,edges_a,edges_b,jaccard_vocab,jaccard_edges,dif_tokens_abs,dif_edges_abs
0,1-11,1,00_Preámbulo.txt,00_Preámbulo.txt,3573,3568,1675,1681,15601,15607,0.970640,0.952208,5,6
1,1-11,2,01_I.txt,01_I.txt,1057,1057,655,653,4987,4988,0.972851,0.952437,0,1
2,1-11,3,02_II.txt,02_II.txt,1116,1120,707,704,5314,5323,0.978962,0.961099,4,9
3,1-11,4,03_III.txt,03_III.txt,1040,1042,638,640,4805,4813,0.984472,0.970498,2,8
4,1-11,5,04_IV.txt,04_IV.txt,1150,1150,711,710,5438,5440,0.970874,0.947368,0,2


In [ ]:
cols = [
    "pareja",
    "unidad",
    "archivo_a",
    "archivo_b",
    "jaccard_vocab",
    "jaccard_edges",
    "dif_edges_abs"
]

df.sort_values(
    by="jaccard_edges",
    ascending=True
)[cols].head(20)

,pareja,unidad,archivo_a,archivo_b,jaccard_vocab,jaccard_edges,dif_edges_abs
62,3-13,14,014.txt,014.txt,0.751295,0.493282,1672
61,3-13,13,013.txt,013.txt,0.778404,0.559702,791
60,3-13,12,012.txt,012.txt,0.800516,0.578334,633
64,3-13,16,016.txt,016.txt,0.812859,0.594136,1765
59,3-13,11,011.txt,011.txt,0.818744,0.600534,473
46,2-12,19,19_XIX.txt,19_CAPITULO_DECIMONOVENO.txt,0.663982,0.608541,3959
63,3-13,15,015.txt,015.txt,0.815789,0.623923,773
66,3-13,18,018.txt,018.txt,0.833228,0.644064,1779
81,4-14,15,15_capítulo_xv.txt,15_capitulo_xv.txt,0.703557,0.647197,436
65,3-13,17,017.txt,017.txt,0.848369,0.659806,276


# LA SIGUIENTE CELDA CREA LOS EDGES NO VOLVER A CARGARLA

# RECUERDA SALTAR LA CELDA ANTERIOR

In [ ]:
BASE = Path("/content/drive/MyDrive/Colab Notebooks/TFM")
CAPITULOS = BASE / "Capítulos"
GRAFOS = BASE / "Grafos"
EDGES_UNIDADES = GRAFOS / "Edges_Unidades"

# BORRAR EDGES ANTERIORES
shutil.rmtree(EDGES_UNIDADES, ignore_errors=True)

# CREAR CARPETA NUEVA VACÍA
EDGES_UNIDADES.mkdir(parents=True, exist_ok=True)

pares = [
    (1,11,"capitulos_libro1","capitulos_libro11"),
    (2,12,"capitulos_libro2","capitulos_libro12"),
    (3,13,"capitulos_libro3","capitulos_libro13"),
    (4,14,"capitulos_libro4","capitulos_libro14"),
    (5,15,"capitulos_libro5","capitulos_libro15"),
    (6,16,"bloques_libro6_v2","bloques_libro16_v2"),
    (7,17,"bloques_libro7","bloques_libro17"),
    (8,18,"bloques_libro8","bloques_libro18"),
    (9,19,"Capitulos_libro9","Capitulos_libro19"),
    (10,20,"bloques_libro10","bloques_libro20"),
]

def guardar_edges_gephi(edges, salida):
    filas = []

    for (a, b), peso in edges.items():
        filas.append({
            "Source": a,
            "Target": b,
            "Weight": peso,
            "Type": "Undirected"
        })

    df_edges = pd.DataFrame(filas)

    df_edges.to_csv(
        salida,
        index=False,
        encoding="utf-8-sig"
    )

total_csv = 0

for libro_a, libro_b, carpeta_a, carpeta_b in pares:
    dir_a = CAPITULOS / carpeta_a
    dir_b = CAPITULOS / carpeta_b

    archivos_a = sorted(dir_a.glob("*.txt"))
    archivos_b = sorted(dir_b.glob("*.txt"))

    salida_par = EDGES_UNIDADES / f"{libro_a}-{libro_b}"
    salida_par.mkdir(parents=True, exist_ok=True)

    if len(archivos_a) != len(archivos_b):
        print(f"REVISAR {libro_a}-{libro_b}: {len(archivos_a)} vs {len(archivos_b)}")
        continue

    for idx, (fa, fb) in enumerate(zip(archivos_a, archivos_b), start=1):
        for libro, archivo in [(libro_a, fa), (libro_b, fb)]:
            texto = archivo.read_text(encoding="utf-8", errors="ignore")

            tokens = tokenizar(texto)
            edges = crear_edges(tokens, ventana=5)

            salida = salida_par / f"unidad_{idx:02d}_libro{libro}_edges.csv"

            guardar_edges_gephi(edges, salida)

            total_csv += 1

    print(f"{libro_a}-{libro_b}: {len(archivos_a)} unidades")

print("\nCSV creados:", total_csv)
print(EDGES_UNIDADES)

1-11: 0 unidades
2-12: 0 unidades
3-13: 0 unidades
4-14: 0 unidades
5-15: 0 unidades
6-16: 0 unidades
7-17: 0 unidades
8-18: 0 unidades
9-19: 0 unidades
10-20: 0 unidades

CSV creados: 0
/content/drive/MyDrive/Colab Notebooks/TFM/Grafos/Edges_Unidades


In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Colab Notebooks/TFM")

print("BASE existe:", BASE.exists())
print()

for p in BASE.iterdir():
    print(p.name)

BASE existe: True

Grafos


In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive")

for p in ROOT.iterdir():
    print(p.name)

Colab Notebooks


In [ ]:
COLAB = ROOT / "Colab Notebooks"

for p in COLAB.iterdir():
    print(p.name)

TFM


In [ ]:
from pathlib import Path

COLAB = Path("/content/drive/MyDrive/Colab Notebooks")

for p in COLAB.iterdir():
    print(p.name)

TFM


In [ ]:
TFM = COLAB / "TFM"

for p in TFM.iterdir():
    print(p.name)

Grafos


In [ ]:
from pathlib import Path

COLAB = Path("/content/drive/MyDrive/Colab Notebooks")

for p in COLAB.rglob("*"):
    if "cap" in p.name.lower():
        print(p)

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Colab Notebooks/TFM")

CAPITULOS = BASE / "Capítulos"

CAPITULOS.mkdir(
    parents=True,
    exist_ok=True
)

print(CAPITULOS)
print("Creada:", CAPITULOS.exists())

/content/drive/MyDrive/Colab Notebooks/TFM/Capítulos
Creada: True


In [ ]:
for p in CAPITULOS.iterdir():
    print(p.name)

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Colab Notebooks/TFM")

for p in BASE.iterdir():
    print(repr(p.name), "->", p)

'Capítulos' -> /content/drive/MyDrive/Colab Notebooks/TFM/Capítulos
'Grafos' -> /content/drive/MyDrive/Colab Notebooks/TFM/Grafos


In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Colab Notebooks/TFM")
CAPITULOS = BASE / "Capítulos"

print("CAPITULOS:", CAPITULOS)
print()

for p in CAPITULOS.iterdir():
    print(p.name)

CAPITULOS: /content/drive/MyDrive/Colab Notebooks/TFM/Capítulos



In [ ]:
for p in CAPITULOS.iterdir():
    print(p.name)

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Colab Notebooks/TFM")
CAPITULOS = BASE / "Capitulos"

print(CAPITULOS)

for p in CAPITULOS.iterdir():
    print(p.name)

/content/drive/MyDrive/Colab Notebooks/TFM/Capitulos


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/TFM/Capitulos'

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Colab Notebooks/TFM")

for p in BASE.iterdir():
    print(repr(p.name))

'Capítulos'
'Grafos'


In [46]:
from google.colab import drive
drive.flush_and_unmount()

Drive not mounted, so nothing to flush and unmount.


In [48]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ValueError: Mountpoint must not already contain files

In [2]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Colab Notebooks/TFM")

print("BASE:", BASE)

for p in BASE.iterdir():
    print(repr(p.name))

CAPITULOS = next(
    p for p in BASE.iterdir()
    if "Cap" in p.name
)

print("\nCAPITULOS:", CAPITULOS)
print("Existe:", CAPITULOS.exists())

print("\nContenido:")
for p in CAPITULOS.iterdir():
    print(p.name)

BASE: /content/drive/MyDrive/Colab Notebooks/TFM
'Capítulos'
'Grafos'

CAPITULOS: /content/drive/MyDrive/Colab Notebooks/TFM/Capítulos
Existe: True

Contenido:
